# Vector Databases & Stores: Core Concepts & Architecture

**Vector Databases** are specialized database engines designed to store, index, and query high-dimensional dense vector embeddings efficiently.

In Retrieval-Augmented Generation (RAG) and LLM application pipelines, vector stores act as semantic long-term memory, allowing systems to retrieve relevant context based on semantic similarity rather than exact keyword matches.

### Indexing & Approximate Nearest Neighbor (ANN) Algorithms

Calculating exact distances across millions of vectors in high-dimensional space is computationally expensive ($O(N)$). Vector databases employ **ANN indexing algorithms** to achieve sub-linear retrieval speed ($O(\log N)$):

1. **Flat Index (Exact Search)**: Compares the query vector against every stored vector. Provides 100% precision/recall but does not scale to large datasets.
2. **Inverted File Index (IVF)**: Partition vector space into clusters (Voronoi cells). Queries only evaluate vectors within nearest cluster centroids.
3. **Hierarchical Navigable Small World (HNSW)**: Graph-based multi-layer indexing structure where upper layers allow fast highway traversal and lower layers provide fine-grained nearest neighbor search.

### Retrieval Strategies

* **Similarity Search**: Retrieves the top-$k$ vectors with smallest distance (or highest similarity score) to the query.
* **Similarity Search with Score**: Returns matching Document objects alongside distance/similarity numerical scores.
* **Maximal Marginal Relevance (MMR)**: Optimizes for both **query relevance** and **diversity** among retrieved chunks to prevent presenting redundant context to the LLM:
  $$\text{MMR} = \arg\max_{d_i \in R \setminus S} \left[ \lambda \text{Sim}_1(d_i, q) - (1 - \lambda) \max_{d_j \in S} \text{Sim}_2(d_i, d_j) \right]$$

### Vector Store Engines Comparison

| Vector Store | Type | Storage / Persistence | Indexing | Typical Use Case |
|---|---|---|---|---|
| **Chroma** | Embedded / Open-Source | In-Memory & Local Disk (sqlite) | HNSW | Rapid prototyping & local RAG |
| **FAISS** | In-Memory Library | Local File (.faiss binary) | IVF, HNSW, Flat | High-speed local indexing |
| **Pinecone** | Cloud Managed | Fully Managed Cloud | Proprietary ANN | Production enterprise cloud deployment |
| **Qdrant** | Production Open-Source | Memory, Local Disk & Cloud | HNSW | High-throughput production microservices |

### LangChain VectorStore Interface & Standard Methods

In LangChain, all vector store integrations (Chroma, FAISS, Pinecone, Qdrant, etc.) inherit from the base VectorStore class (langchain_core.vectorstores.VectorStore). This provides a unified API across all vector databases.

#### 1. Creation & Initialization Methods
* from_documents(documents, embedding, **kwargs): Instantiates a new vector store and populates it directly from a list of Document objects.
* from_texts(texts, embedding, metadatas=None, **kwargs): Creates a new vector store directly from raw text strings and optional metadata dictionaries.

#### 2. Data Ingestion & Mutation Methods
* add_documents(documents, ids=None, **kwargs): Ingests additional Document objects into an existing vector store. Returns assigned document IDs.
* add_texts(texts, metadatas=None, ids=None, **kwargs): Ingests additional raw text strings and metadata dictionaries.
* update_document(document_id, document): Updates stored text content and metadata for a specific document ID.
* delete(ids=None, **kwargs): Removes vectors and metadata matching the specified list of document IDs.

#### 3. Search & Retrieval Methods
* similarity_search(query, k=4, filter=None, **kwargs): Returns top-$k$ Document objects closest in semantic distance to the query text.
* similarity_search_with_score(query, k=4, filter=None, **kwargs): Returns a list of (Document, distance_score) tuples.
* similarity_search_by_vector(embedding, k=4, filter=None, **kwargs): Queries the vector index using a pre-computed vector embedding list.
* max_marginal_relevance_search(query, k=4, fetch_k=20, lambda_mult=0.5, filter=None): Performs MMR search to balance query relevance ($\lambda$) and result diversity ($1-\lambda$).
* max_marginal_relevance_search_by_vector(embedding, k=4, fetch_k=20, lambda_mult=0.5, filter=None): Executes MMR retrieval starting from a query vector.

#### 4. LCEL Retriever Conversion
* as_retriever(search_type="similarity", search_kwargs={"k": 4}): Converts the vector store into a VectorStoreRetriever runnable for LCEL pipeline composition.

### Vector Store Comparison Matrix

| Vector Store | Deployment / Hosting | Indexing Engine | Primary Use Case |
|---|---|---|---|
| **Chroma DB** | Local / Ephemeral | HNSW | Prototyping, local development |
| **FAISS** | In-Memory / File | Inverted Index / Flat | Fast CPU/GPU local similarity search |
| **Pinecone** | Managed Cloud | Proprietary HNSW | Managed enterprise vector search |
| **Qdrant** | Self-Hosted / Cloud | HNSW with Payload Filtering | Filtering-heavy production apps |
| **Astra DB** | DataStax Cassandra Cloud | Jvector / Cassandra Vector | Massive scale serverless Cassandra vector search |